# US Core Profile Intro Generator

Generates the intro markdown for a US Core profile page from a
`StructureDefinition` JSON snapshot and a Jinja2 template.

**Inputs**
- `SD_PATH`: path to a StructureDefinition JSON file (post-snapshot, from the IG Publisher's first build)
- `TEMPLATE_PATH`: path to the Jinja2 template

**Output**: rendered markdown printed inline at the end.

Edit the paths in the *Inputs* cell, then Run All.

## Imports and constants

In [31]:
import json
import re
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()  # reads .env into os.environ
from anthropic import Anthropic

from jinja2 import Environment, FileSystemLoader, StrictUndefined

USCDI_EXTENSION_URL = (
    "http://hl7.org/fhir/us/core/StructureDefinition/uscdi-requirement"
)
W5_AUTHOR_FRAGMENT = "author"
VOWELS = set("aeiou")

## Inputs

Edit these two paths to point at your snapshot and template.

In [32]:
SD_PATH = Path("/Users/ehaas/Documents/FHIR/USCDIV7/output/StructureDefinition-us-core-appointment.json")
TEMPLATE_PATH = Path("/Users/ehaas/Documents/FHIR/USCDIV7/Profile_intro.md.j2")

## Element classification

Pure functions that examine one element dict and return a flag.

In [33]:
def has_uscdi_extension(element):
    for ext in element.get("extension", []) or []:
        if ext.get("url") == USCDI_EXTENSION_URL:
            return True
    return False


def has_w5_author_mapping(element):
    for mapping in element.get("mapping", []) or []:
        if (
            mapping.get("identity") == "w5"
            and W5_AUTHOR_FRAGMENT in (mapping.get("map") or "")
        ):
            return True
    return False


def is_root(element, resource_type):
    return element.get("path") == resource_type


def is_must_have(element, resource_type):
    return (
        not is_root(element, resource_type)
        and element.get("mustSupport") is True
        and (element.get("min") or 0) >= 1
    )


def is_must_support(element, resource_type):
    return (
        not is_root(element, resource_type)
        and element.get("mustSupport") is True
        and (element.get("min") or 0) == 0
        and not has_uscdi_extension(element)
    )


def is_uscdi(element, resource_type):
    return (
        not is_root(element, resource_type)
        and has_uscdi_extension(element)
    )

## Label formatting

Produces "a/an &lt;short&gt;" from an element's `short` definition.

In [34]:
def is_pipe_separated_codes(short):
    """True if short is a list of pipe-separated codes (e.g. 'active | inactive')."""
    if "|" not in short:
        return False
    parts = [p.strip() for p in short.split("|")]
    return all(p and " " not in p for p in parts)


def humanize_element_name(path):
    """Return the last path segment as space-separated lowercase words."""
    last = path.rsplit(".", 1)[-1]
    spaced = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", last)
    return spaced.lower()


def format_label(element):
    """Return the bullet text for an element.

    If `short` is a pipe-separated code list, use the humanized element name
    prefixed with 'a'/'an' (max == '1') or 'one or more' (max != '1').
    Otherwise return `short` with a lowercased first character.
    """
    short = element.get("short") or element.get("path", "")
    short = short.strip().rstrip(".")
    if not short:
        return ""

    if is_pipe_separated_codes(short):
        name = humanize_element_name(element.get("path", ""))
        if not name:
            return ""
        max_val = str(element.get("max", "1"))
        if max_val == "1":
            article = "an" if name[:1] in "aeiou" else "a"
            return f"{article} {name}"
        return f"one or more {name}"

    return short[0].lower() + short[1:]


## Build the render context

Walks the snapshot and assembles the Jinja context dict.

In [35]:
def build_context(sd):
    resource_type = sd.get("type", "")
    elements = (sd.get("snapshot") or {}).get("element", []) or []

    must_have = []
    must_support = []
    uscdi = []
    has_author = False

    for el in elements:
        if has_w5_author_mapping(el):
            has_author = True

        label = format_label(el)
        if not label:
            continue

        item = {"label": label, "footnote": None}

        if is_must_have(el, resource_type):
            must_have.append(item)
        elif is_uscdi(el, resource_type):
            uscdi.append(item)
        elif is_must_support(el, resource_type):
            must_support.append(item)

    return {
        "file_name": SD_PATH.stem,
        "resource_type": resource_type,
        "must_have": must_have,
        "must_support": must_support,
        "uscdi": uscdi,
        "has_uscdi": bool(uscdi),
        "has_author": has_author,
    }

## Render

Loads the template and substitutes the context.

In [36]:
def render(template_path, context):
    template_path = Path(template_path)
    env = Environment(
        loader=FileSystemLoader(str(template_path.parent)),
        undefined=StrictUndefined,
        keep_trailing_newline=True,
    )
    template = env.get_template(template_path.name)
    return template.render(**context)

## Load the StructureDefinition

In [37]:
with SD_PATH.open() as f:
    sd = json.load(f)

print(f"Loaded {SD_PATH.name}")
print(f"  resource type: {sd.get('type')}")
print(f"  snapshot elements: {len(sd.get('snapshot', {}).get('element', []))}")

Loaded StructureDefinition-us-core-appointment.json
  resource type: Appointment
  snapshot elements: 39


## Inspect the classified context

Useful for spot-checking which elements landed in which list before
rendering.

In [38]:
context = build_context(sd)
print(f"file_name : {context['file_name']}")
print(f"resource_type : {context['resource_type']}")
print(f"has_uscdi     : {context['has_uscdi']}")
print(f"has_author    : {context['has_author']}")
print()
print("Must Have:")
for item in context["must_have"]:
    print(f"  - {item['label']}")
print()
print("Must Support:")
for item in context["must_support"]:
    print(f"  - {item['label']}")
print()
print("USCDI:")
for item in context["uscdi"]:
    print(f"  - {item['label']}")

file_name : StructureDefinition-us-core-appointment
resource_type : Appointment
has_uscdi     : False
has_author    : False

Must Have:
  - a status
  - person, Location/HealthcareService or Device
  - a status

Must Support:
  - external Ids for this item
  - the specific service that is to be performed during this appointment
  - when appointment is to take place
  - when appointment is to conclude
  - participants involved in appointment

USCDI:


## Render and display

The cell below renders the template and displays the result as Markdown.
The output is exactly what would be written to
`input/intro-notes/StructureDefinition-<id>-intro.md`.

In [39]:
from IPython.display import Markdown, display

output = render(TEMPLATE_PATH, context)
display(Markdown(output))

**Example Usage Scenarios:**

The following are example usage scenarios for the US Core Appointment profile:

-   Query for Appointment resources belonging to a Patient
-   [Record or update]  a Patient Appointment

### Mandatory and Must Support Data Elements

The following data elements must always be present ([Mandatory] definition) or must be supported if the data is present in the sending system ([Must Support] definition). They are presented below in a simple human-readable explanation. Profile specific guidance and examples are provided as well. The [Formal Views] below provides the formal summary, definitions, and terminology requirements.

**Each Appointment Must Have:**


1. a status
1. person, Location/HealthcareService or Device
1. a status

**Each Appointment Must Support:**


1. external Ids for this item
1. the specific service that is to be performed during this appointment
1. when appointment is to take place
1. when appointment is to conclude
1. participants involved in appointment



### Profile Specific Implementation Guidance

This section provides detailed implementation guidance for the US Core Profile to support implementation and certification.






{% include structure-table-block.md file_name="StructureDefinition-us-core-appointment" %}

{% include link-list.md %}


## Raw output

The same rendered text without Markdown interpretation, useful for
copy-paste or diffing.

In [40]:
print(output)

**Example Usage Scenarios:**

The following are example usage scenarios for the US Core Appointment profile:

-   Query for Appointment resources belonging to a Patient
-   [Record or update]  a Patient Appointment

### Mandatory and Must Support Data Elements

The following data elements must always be present ([Mandatory] definition) or must be supported if the data is present in the sending system ([Must Support] definition). They are presented below in a simple human-readable explanation. Profile specific guidance and examples are provided as well. The [Formal Views] below provides the formal summary, definitions, and terminology requirements.

**Each Appointment Must Have:**


1. a status
1. person, Location/HealthcareService or Device
1. a status

**Each Appointment Must Support:**


1. external Ids for this item
1. the specific service that is to be performed during this appointment
1. when appointment is to take place
1. when appointment is to conclude
1. participants involved 

## Polish labels with Claude (optional)

FHIR `short` definitions are often terse or awkwardly phrased — "when appointment is to conclude" rather than "when an appointment ends." The function below batches all bullets in the current context through the Anthropic API and asks Claude to rewrite them as more natural English phrases, while preserving any leading article (`a`, `an`, `one or more`) emitted by `format_label`.

Requires:
- `pip install anthropic`
- `ANTHROPIC_API_KEY` set in the environment (or passed explicitly).

Skip these cells if you don't want LLM polish — the rest of the notebook works without them.

In [41]:
POLISH_PROMPT = """You are editing bullet text from a US Core FHIR Implementation Guide. Each bullet describes one data element of the {resource_type} profile and appears under headings like "Each {resource_type} Must Have:" or "Each {resource_type} Must Support:".

Rewrite each `label` as a more natural English phrase that reads well in a bulleted list.

Rules:
- If the label starts with "a ", "an ", or "one or more ", keep that prefix exactly as-is. Do not add a leading article to labels that lack one.
- Preserve meaning exactly. Do not invent details not present in the original.
- Use plain, present-tense English. Prefer common words over jargon.
- Aim for under 12 words. Concision wins.
- Do not capitalize the first letter (these sit mid-sentence under a heading).
- No trailing punctuation.
- If the original already reads naturally, return it unchanged.

Examples:
- "when appointment is to conclude" -> "when an appointment ends"
- "code that tells you what the patient is allergic to" -> "code identifying the allergy"
- "a clinical status" -> "a clinical status"
- "one or more category" -> "one or more categories"

Return ONLY a JSON array, no preamble or commentary, shaped like:
[{{"list": "must_have", "index": 0, "polished": "..."}}, ...]

Bullets to polish:
{items_json}
"""


def polish_labels(context, model="claude-sonnet-4-6", api_key=None, verbose=True):
    """Rewrite bullet labels via the Anthropic API; return a new context dict."""
    from anthropic import Anthropic

    items = []
    for list_name in ("must_have", "must_support", "uscdi"):
        for idx, item in enumerate(context[list_name]):
            items.append({"list": list_name, "index": idx, "label": item["label"]})

    if not items:
        return {**context}

    prompt = POLISH_PROMPT.format(
        resource_type=context["resource_type"],
        items_json=json.dumps(items, indent=2),
    )

    client = Anthropic(api_key=api_key) if api_key else Anthropic()
    if verbose:
        print(f"Calling {model} with {len(items)} bullets...")

    response = client.messages.create(
        model=model,
        max_tokens=2000,
        messages=[{"role": "user", "content": prompt}],
    )

    raw = response.content[0].text.strip()
    # Strip fenced code blocks if the model added them despite the instructions.
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw, flags=re.MULTILINE)

    try:
        polished = json.loads(raw)
    except json.JSONDecodeError as e:
        print("Could not parse model output as JSON. Raw response:")
        print(raw)
        raise

    new_context = {**context}
    for list_name in ("must_have", "must_support", "uscdi"):
        new_context[list_name] = [dict(item) for item in context[list_name]]

    for entry in polished:
        new_context[entry["list"]][entry["index"]]["label"] = entry["polished"]

    return new_context

### Run the polish step and show before/after

Compares the original labels to the polished ones side by side. Unchanged labels are still listed for transparency.

In [42]:
polished_context = polish_labels(context)

for list_name in ("must_have", "must_support", "uscdi"):
    if not context[list_name]:
        continue
    print(f"\n{list_name}:")
    width = max(len(item["label"]) for item in context[list_name])
    for orig, new in zip(context[list_name], polished_context[list_name]):
        marker = "  " if orig["label"] == new["label"] else "->"
        print(f"  {orig['label']:<{width}}  {marker}  {new['label']}")

Calling claude-sonnet-4-6 with 8 bullets...

must_have:
  a status                                          a status
  person, Location/HealthcareService or Device  ->  person, location, healthcare service, or device
  a status                                          a status

must_support:
  external Ids for this item                                            ->  external identifiers for this appointment
  the specific service that is to be performed during this appointment  ->  the specific service to be performed during the appointment
  when appointment is to take place                                     ->  when the appointment starts
  when appointment is to conclude                                       ->  when the appointment ends
  participants involved in appointment                                  ->  participants involved in the appointment


### Render with polished labels

In [43]:
polished_output = render(TEMPLATE_PATH, polished_context)
Markdown(polished_output)

**Example Usage Scenarios:**

The following are example usage scenarios for the US Core Appointment profile:

-   Query for Appointment resources belonging to a Patient
-   [Record or update]  a Patient Appointment

### Mandatory and Must Support Data Elements

The following data elements must always be present ([Mandatory] definition) or must be supported if the data is present in the sending system ([Must Support] definition). They are presented below in a simple human-readable explanation. Profile specific guidance and examples are provided as well. The [Formal Views] below provides the formal summary, definitions, and terminology requirements.

**Each Appointment Must Have:**


1. a status
1. person, location, healthcare service, or device
1. a status

**Each Appointment Must Support:**


1. external identifiers for this appointment
1. the specific service to be performed during the appointment
1. when the appointment starts
1. when the appointment ends
1. participants involved in the appointment



### Profile Specific Implementation Guidance

This section provides detailed implementation guidance for the US Core Profile to support implementation and certification.






{% include structure-table-block.md file_name="StructureDefinition-us-core-appointment" %}

{% include link-list.md %}


## Raw polished output

The same rendered text without Markdown interpretation, useful for

copy-paste or diffing.

In [44]:
print(polished_output)

**Example Usage Scenarios:**

The following are example usage scenarios for the US Core Appointment profile:

-   Query for Appointment resources belonging to a Patient
-   [Record or update]  a Patient Appointment

### Mandatory and Must Support Data Elements

The following data elements must always be present ([Mandatory] definition) or must be supported if the data is present in the sending system ([Must Support] definition). They are presented below in a simple human-readable explanation. Profile specific guidance and examples are provided as well. The [Formal Views] below provides the formal summary, definitions, and terminology requirements.

**Each Appointment Must Have:**


1. a status
1. person, location, healthcare service, or device
1. a status

**Each Appointment Must Support:**


1. external identifiers for this appointment
1. the specific service to be performed during the appointment
1. when the appointment starts
1. when the appointment ends
1. participants involved in 

## Write to the intro-notes folder using

In [45]:
out_path = Path(r'/Users/ehaas/Documents/FHIR/USCDIV7/input/intro-notes')
out_file = out_path / f"{SD_PATH.stem}-intro.md"
print(f"saving to {out_file} ...")
out_file.write_text(polished_output)

saving to /Users/ehaas/Documents/FHIR/USCDIV7/input/intro-notes/StructureDefinition-us-core-appointment-intro.md ...


1314